In [ ]:
import torch
import torch.nn as nn

In [ ]:
class LinearAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        # Input N, 784
        self.encoder = nn.Sequential(
            nn.Linear(28*28, 128), # N, 784 -> N, 128
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 12),
            nn.ReLU(),
            nn.Linear(12, 3), # -> N, 3
        )

        self.decoder = nn.Sequential(
            nn.Linear(3,12),
            nn.ReLU(),
            nn.Linear(12,64),
            nn.ReLU(),
            nn.Linear(64,128),
            nn.ReLU(),
            nn.Linear(128,28*28),
            # Our images have values between 0 and 1, 
            # so our activation should map to 0,1 -> sigmoid
            nn.Sigmoid(),
        )

    def forward(self, x):
        enc = self.encoder(x)
        dec = self.decoder(enc)
        return dec
    
    

In [ ]:
class CNNAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        # Input N, 1, 28, 28
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, 3, stride=2, padding=1), # N, 16, 14, 14
            nn.ReLU(),
            nn.Conv2d(16, 32, 3, stride=2, padding=1), # N, 32, 7, 7
            nn.ReLU(),
            nn.Conv2d(32, 64, 7) # N, 64, 1, 1
        )

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 7),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 16, 3, stride=2, padding=1, output_padding=1), # N, 16, 14, 14
            nn.ReLU(),
            nn.ConvTranspose2d(16, 1, 3, stride=2, padding=1, output_padding=1), # N, 1, 28, 28,
            nn.Sigmoid()
        )

    def forward(self, x):
        enc = self.encoder(x)
        dec = self.decoder(enc)
        return dec
# nn.MaxPool2d - Revert: nn.MaxUnpool2d

# VAE

In [ ]:
import torch
import torch.nn.functional as F
import torch.nn as nn

class VAE(nn.Module):
    def __init__(self, latent_dim, in_shape=(3,64,64)):
        super().__init__()
        self.latent_dim = latent_dim
        self.in_shape = in_shape
        
        # -- ENCODER --
        # Input:  N, 3, 64, 64
        # Output: N, 2*latent_dim (mu, log_sigma)
        self.encoder = nn.Sequential(
            nn.Conv2d(self.in_shape[0], 32, 4, stride=2), # -> N, 32, 31, 31
            nn.ReLU(),
            nn.Conv2d(32, 64, 4, stride=2), # -> N, 64, 14, 14
            nn.ReLU(),
            nn.Conv2d(64, 128, 4, stride=2), # -> N,128,6,6
            nn.ReLU(),
            nn.Conv2d(128, 256,4, stride=2), # -> N,256,2,2
            nn.ReLU(),
            nn.Flatten(start_dim=1), # N, 256*2*2 = 1024
            nn.Linear(256*2*2, 2*self.latent_dim), # N,1024 -> N, 2*latent_dim
        )

        # -- DECODER --
        # Input: N, latent_dim
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 256*2*2), # (N, 256*2*2)
            nn.Unflatten(1, (1024, 1, 1)), # (N, 1024, 1, 1)
            nn.ConvTranspose2d(1024, 128, 5, stride=2), # -> (N, 128, 5, 5)
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 5, stride=2), # -> N, 64, 13, 13
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 6, stride=2), # -> N, 32, 30, 30
            nn.ReLU(),
            nn.ConvTranspose2d(32, 3, 6, stride=2), # -> N, 3, 64, 64
            nn.Sigmoid()
        )

    def forward(self, x):
        # encoder
        mu, log_var = torch.split(self.encoder(x), self.latent_dim, dim=1) # (N, latent_dim), (N, latent_dim)

        # reparameterization trick
        eps = torch.randn_like(log_var)
        z = mu + torch.exp(0.5 * log_var) * eps # -> latent_dim

        # decoder
        recon_x =  self.decoder(z)

        return recon_x, mu, log_var
    
def vae_loss(recon_x, x, mu, log_var):
    reconstruction_loss = F.mse_loss(recon_x, x, reduction='sum')
    kl_loss = -0.5 * torch.sum(1 + log_var - mu.pow(2) - torch.exp(log_var)) 
    loss = reconstruction_loss + kl_loss
    return loss

# train

In [ ]:
from torchvision import datasets, transforms


In [ ]:
import torch
import torchvision.transforms as transforms
import torchvision.datasets as datasets

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Define a transform to resize images to 64x64
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
])

# Replace MNIST with CIFAR10
# Pass the 'transform' variable defined above, not the 'transforms' module
cifar_data = datasets.CIFAR10(root='cifar10', train=True, download=True, transform=transform)
data_loader = torch.utils.data.DataLoader(
    dataset=cifar_data, # Use the new dataset variable
    batch_size=64,
    shuffle=True,
)

dataiter = iter(data_loader)
images, labels = next(dataiter)
print(torch.min(images), torch.max(images))
# Check image shape (should have 3 channels for RGB)
print(images.shape)

In [ ]:
# Count total parameters
model = VAE(latent_dim=32)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params}")

# Count trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total number of trainable parameters: {trainable_params}")

In [ ]:
model= VAE(latent_dim=2)
optim = torch.optim.Adam(model.parameters(),lr=1e-3)

In [ ]:
n_epochs = 1
outputs = []

# --- Training Loop ---
for epoch in range(n_epochs):
    epoch_loss = 0.0
    num_batches = 0
    for i, (img, _) in enumerate(data_loader): # iterate batches
        # if i >= 999: # train on partial data
            # break

        img = img.to(device)

        # --- VAE Specific ---
        recon_x, mu, log_var = model(img)
        loss = vae_loss(recon_x, img, mu, log_var)
        print(f'loss: {loss}')
        # --- End VAE Specific ---

        # --- General Autoencoder (Uncomment if using Linear/CNN) ---
        # img = img.view(img.size(0), -1) # Flatten for Linear AE
        # recon_x = model(img)
        # loss = F.mse_loss(recon_x, img, reduction='sum') # Example loss for Linear AE
        # --- End General Autoencoder ---

        optim.zero_grad()
        loss.backward()
        optim.step()

        epoch_loss += loss.item()
        num_batches += 1

        # Store results from the first batch of each epoch for visualization
        if i == 0:
            outputs.append((epoch, img.detach().cpu(), recon_x.detach().cpu()))

    avg_epoch_loss = epoch_loss / (num_batches * data_loader.batch_size) # More representative loss
    print(f'Epoch: {epoch+1}, Average Loss: {avg_epoch_loss:.4f}')

# visualize

In [ ]:
# --- Visualization ---
import matplotlib.pyplot as plt

selected_epochs_indices = [0, n_epochs // 2, n_epochs - 1]  # Indices corresponding to first, middle, last epochs
n_images = 10  # Number of images to display

print("\nVisualizing results from selected epochs...")
for epoch_idx in selected_epochs_indices:
    if epoch_idx < len(outputs): # Check if the epoch data exists
        epoch, original_images, reconstructed_images = outputs[epoch_idx]

        fig, axes = plt.subplots(2, n_images, figsize=(20, 4))

        # Display original images in the first row
        for i in range(min(n_images, original_images.shape[0])): # Handle cases with fewer images than n_images
            axes[0, i].imshow(original_images[i].squeeze().numpy(), cmap='gray')
            axes[0, i].set_title(f'Original')
            axes[0, i].axis('off')

        # Display reconstructed images in the second row
        for i in range(min(n_images, reconstructed_images.shape[0])):
            axes[1, i].imshow(reconstructed_images[i].squeeze().numpy(), cmap='gray')
            axes[1, i].set_title(f'Reconstructed')
            axes[1, i].axis('off')

        # Hide unused subplots if any
        for i in range(min(n_images, original_images.shape[0]), n_images):
             axes[0, i].axis('off')
             axes[1, i].axis('off')

        plt.suptitle(f'Epoch {epoch+1} Results')
        # Adjust layout to prevent title overlap
        fig.tight_layout(rect=[0, 0.03, 1, 0.95])
        plt.show()
    else:
        print(f"Warning: Epoch {epoch_idx+1} data not available for visualization (total epochs: {len(outputs)}).")